<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/ltm_in_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Using existing memories

In [2]:
!pip install -U langchain-openai

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from google.colab import userdata

In [10]:
api_key = userdata.get("OPENAI_API_KEY")
model = ChatOpenAI(api_key=api_key, model="gpt-4o-mini")

In [12]:
# LTM store
store = InMemoryStore()

namespace = ("user", "u1", "details")

# seed memories before running graph
james_bond = {
    "profile_1": {"data": "Name: James Bond"},
    "profile_2": {"data": "Profession: MI6 Secret Agent"},
    "profile_3": {"data": "Code Name: 007"},
    "preference_1": {"data": "Prefers Martinis, shaken not stirred"},
    "preference_2": {"data": "Likes Aston Martin cars"},
    "preference_3": {"data": "Enjoys high-tech gadgets"},
    "project_1": {"data": "Mission: Protect British national security"},
    "project_2": {"data": "Specialization: Espionage and covert operations"}
}

for key, value in james_bond.items():
  store.put(namespace, key, {"data": value})

In [8]:
memories = store.search(namespace)
for memory in memories:
  print(memory.value)

{'data': {'data': 'Name: James Bond'}}
{'data': {'data': 'Profession: MI6 Secret Agent'}}
{'data': {'data': 'Code Name: 007'}}
{'data': {'data': 'Prefers Martinis, shaken not stirred'}}
{'data': {'data': 'Likes Aston Martin cars'}}
{'data': {'data': 'Enjoys high-tech gadgets'}}
{'data': {'data': 'Mission: Protect British national security'}}
{'data': {'data': 'Specialization: Espionage and covert operations'}}


In [9]:
# system prompt
SYSTEM_PROMPT = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize your responses based
on what you know about the user.

Your goal is to provide relevant, friendly, and tailored assistance that reflects
the user's preferences, context, and past interactions.

If the user's name or relevant personal context is available, always personalize
your responses by:
  - Always address the user by their name (e.g., "Sure, James...") when appropriate
  - Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible.

Use personalization especially in:
  - Greetings and transitions
  - Help or tailored guidance
  - Follow-up messages that continue from the past context

Always ensure that personalization is based only on know user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile.

The user's memory (which may be empty) is provided as: {user_details}
"""

In [13]:
def chat_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
  user_id = config["configurable"]["user_id"]

  # fetch user details
  namespace = ("user", user_id, "details")
  memories = store.search(namespace)

  # convert memory items into string blob
  if memories:
    user_details = "\n".join(f"- {memory.value.get("data", "")}" for memory in memories)
  else:
    user_details = ""

  system_prompt = SYSTEM_PROMPT.format(user_details=user_details)

  system_msg = SystemMessage(content=system_prompt)

  response = model.invoke([system_msg] + state["messages"])
  return {"messages": [response]}

In [14]:
graph = StateGraph(MessagesState)
graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

workflow = graph.compile(store=store)

In [18]:
# invoke workflow
config = {"configurable": {"user_id": "u1"}}

result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Who am I and what are my preferences?"}]},
    config
)
print(result["messages"][-1].content)

You are James Bond, also known by your code name 007. You work as a secret agent for MI6, specializing in espionage and covert operations to protect British national security. You have a taste for Martinis, specifically shaken not stirred, and you appreciate high-tech gadgets. You also have a fondness for Aston Martin cars. 

Is there anything specific you would like assistance with today, James? 

Here are a few questions to consider: 
1. Are you looking for advice on your next mission or gadget?
2. Would you like recommendations for the best Aston Martin models?
3. Do you need tips on how to mix the perfect Martini?
